In [1]:

import pandas as pd

# Dataset A: High-concentration portfolio
portfolio_a = pd.DataFrame([
    {
        "ticker": "SBER",
        "name": "Sberbank",
        "quantity": 15000,
        "current_price": 305.50,
        "total_value": 4582500,
        "weight": 45.825,
        "currency": "RUB",
    },
    {
        "ticker": "GAZP",
        "name": "Gazprom",
        "quantity": 20000,
        "current_price": 130.20,
        "total_value": 2604000,
        "weight": 26.040,
        "currency": "RUB",
    },
    {
        "ticker": "LKOH",
        "name": "Lukoil",
        "quantity": 300,
        "current_price": 6850.00,
        "total_value": 2055000,
        "weight": 20.550,
        "currency": "RUB",
    },
    {
        "ticker": "RUB",
        "name": "Russian Ruble Cash",
        "quantity": 758500,
        "current_price": 1.00,
        "total_value": 758500,
        "weight": 7.585,
        "currency": "RUB",
    },
])

# Dataset B: More diversified portfolio
portfolio_b = pd.DataFrame([
    {
        "ticker": "SU26238RMFS4",
        "name": "Russian Government Bond",
        "quantity": 5000,
        "current_price": 540.20,
        "total_value": 2701000,
        "weight": 27.010,
        "currency": "RUB",
    },
    {
        "ticker": "SU29014RMFS6",
        "name": "Russian Government Bond",
        "quantity": 2500,
        "current_price": 985.00,
        "total_value": 2462500,
        "weight": 24.625,
        "currency": "RUB",
    },
    {
        "ticker": "LQDT",
        "name": "Liquidity Fund",
        "quantity": 1200000,
        "current_price": 1.52,
        "total_value": 1824000,
        "weight": 18.240,
        "currency": "RUB",
    },
    {
        "ticker": "SBER",
        "name": "Sberbank",
        "quantity": 5000,
        "current_price": 305.50,
        "total_value": 1527500,
        "weight": 15.275,
        "currency": "RUB",
    },
    {
        "ticker": "TGLD",
        "name": "Gold Fund",
        "quantity": 150000,
        "current_price": 9.90,
        "total_value": 1485000,
        "weight": 14.850,
        "currency": "RUB",
    },
])

print("DATASET A — HIGH CONCENTRATION")
display(portfolio_a)

print("DATASET B — MORE DIVERSIFIED")
display(portfolio_b)

print("Dataset A total value:", portfolio_a["total_value"].sum())
print("Dataset B total value:", portfolio_b["total_value"].sum())
print("Dataset A weight total:", portfolio_a["weight"].sum())
print("Dataset B weight total:", portfolio_b["weight"].sum())


DATASET A — HIGH CONCENTRATION


,ticker,name,quantity,current_price,total_value,weight,currency
0,SBER,Sberbank,15000,305.5,4582500,45.825,RUB
1,GAZP,Gazprom,20000,130.2,2604000,26.040,RUB
2,LKOH,Lukoil,300,6850.0,2055000,20.550,RUB
3,RUB,Russian Ruble Cash,758500,1.0,758500,7.585,RUB


DATASET B — MORE DIVERSIFIED


,ticker,name,quantity,current_price,total_value,weight,currency
0,SU26238RMFS4,Russian Government Bond,5000,540.20,2701000,27.010,RUB
1,SU29014RMFS6,Russian Government Bond,2500,985.00,2462500,24.625,RUB
2,LQDT,Liquidity Fund,1200000,1.52,1824000,18.240,RUB
3,SBER,Sberbank,5000,305.50,1527500,15.275,RUB
4,TGLD,Gold Fund,150000,9.90,1485000,14.850,RUB


Dataset A total value: 10000000
Dataset B total value: 10000000
Dataset A weight total: 100.0
Dataset B weight total: 100.0


In [5]:

def validate_portfolio(df, portfolio_name):
    """Check the basic integrity of a benchmark portfolio."""

    print(f"\nChecking {portfolio_name}...")

    assert not df.isnull().any().any(), (
        f"{portfolio_name}: missing values detected"
    )

    assert (df["quantity"] >= 0).all(), (
        f"{portfolio_name}: negative quantity detected"
    )

    assert (df["current_price"] >= 0).all(), (
        f"{portfolio_name}: negative price detected"
    )

    assert (df["total_value"] >= 0).all(), (
        f"{portfolio_name}: negative holding value detected"
    )

    calculated_values = df["quantity"] * df["current_price"]

    assert (
        calculated_values.round(2)
        == df["total_value"].round(2)
    ).all(), (
        f"{portfolio_name}: quantity × price does not match value"
    )

    assert abs(df["weight"].sum() - 100.0) < 0.001, (
        f"{portfolio_name}: weights do not total 100%"
    )

    assert abs(df["total_value"].sum() - 10_000_000) < 0.01, (
        f"{portfolio_name}: portfolio total is incorrect"
    )

    print("PASS: All benchmark data checks succeeded.")


validate_portfolio(portfolio_a, "Dataset A")
validate_portfolio(portfolio_b, "Dataset B")



Checking Dataset A...
PASS: All benchmark data checks succeeded.

Checking Dataset B...
PASS: All benchmark data checks succeeded.


In [6]:

import numpy as np


def calculate_hhi(weights):
    """Calculate HHI using portfolio weights in percentage points."""

    weights = np.asarray(weights, dtype=float)

    if weights.ndim != 1 or len(weights) == 0:
        raise ValueError("Weights must be a non-empty one-dimensional array.")

    if not np.isfinite(weights).all():
        raise ValueError("Weights must contain only finite values.")

    if (weights < 0).any():
        raise ValueError("Negative weights are not supported by this HHI function.")

    if not np.isclose(weights.sum(), 100.0, atol=0.001):
        raise ValueError("Portfolio weights must sum to 100%.")

    return float(np.sum(weights ** 2))


def classify_hhi(hhi):
    """Apply the concentration thresholds specified by Aegis."""

    if hhi < 1500:
        return "Low"
    elif hhi < 2500:
        return "Medium"
    else:
        return "High"


results = []

for portfolio_name, portfolio in [
    ("Dataset A", portfolio_a),
    ("Dataset B", portfolio_b),
]:
    hhi = calculate_hhi(portfolio["weight"])

    results.append({
        "Portfolio": portfolio_name,
        "HHI": round(hhi, 2),
        "Classification": classify_hhi(hhi),
    })

hhi_results = pd.DataFrame(results)
display(hhi_results)


,Portfolio,HHI,Classification
0,Dataset A,3257.85,High
1,Dataset B,2122.48,Medium


In [7]:

expected_hhi = {
    "Dataset A": 3257.85,
    "Dataset B": 2122.48,
}

actual_hhi = {
    "Dataset A": calculate_hhi(portfolio_a["weight"]),
    "Dataset B": calculate_hhi(portfolio_b["weight"]),
}

for portfolio_name, expected in expected_hhi.items():
    actual = actual_hhi[portfolio_name]

    assert np.isclose(actual, expected, atol=0.01), (
        f"{portfolio_name}: expected {expected}, got {actual}"
    )

print("PASS: Both HHI calculations match the official benchmarks.")


PASS: Both HHI calculations match the official benchmarks.


In [8]:
from pathlib import Path

# Project root: Aegis Wealth Terminal
PROJECT_ROOT = Path.cwd().parent

print("Current notebook directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nPotential benchmark and price files:")

search_terms = (
    "benchmark",
    "price",
    "return",
    "portfolio",
    "market",
    "mcftr",
    "historical",
    "dataset",
)

found_files = []

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    # Avoid searching inside Git and virtual-environment folders.
    if any(
        part in {".git", ".venv", "node_modules"}
        for part in path.parts
    ):
        continue

    if path.suffix.lower() in {
        ".csv", ".xlsx", ".xls", ".parquet", ".json"
    }:
        if any(term in path.name.lower() for term in search_terms):
            found_files.append(path)

if found_files:
    for path in found_files:
        print(path.relative_to(PROJECT_ROOT))
else:
    print("No matching benchmark files found.")

Current notebook directory:
C:\Users\NEW USER\AegisWealthTerminal\research

Project root:
C:\Users\NEW USER\AegisWealthTerminal

Potential benchmark and price files:
No matching benchmark files found.
